In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
target_ticker = "GC=F"

gold = yf.download(
    target_ticker,
    period="5y",
    interval="1d",
    auto_adjust=False
)

gold.head()

In [ ]:
gold.shape

In [ ]:
gold.info()

In [ ]:
gold.isnull().sum()

In [ ]:
df_gold = gold.copy()

# Flatten MultiIndex columns from yfinance
df_gold.columns = df_gold.columns.get_level_values(0)

df_gold.head()

In [ ]:
df_gold.columns

In [ ]:
df_gold = df_gold[["Open", "High", "Low", "Close", "Volume"]]

df_gold.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_gold.index, df_gold["Close"])
plt.title("Gold Futures Closing Price - Last 5 Years")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True)
plt.show()

In [ ]:
df_gold.describe()

In [ ]:
zero_volume_count = (df_gold["Volume"] == 0).sum()
total_rows = len(df_gold)

print("Zero volume rows:", zero_volume_count)
print("Total rows:", total_rows)
print("Zero volume percentage:", round((zero_volume_count / total_rows) * 100, 2), "%")

In [ ]:
df_gold_model = df_gold[["Open", "High", "Low", "Close", "Volume"]].copy()

df_gold_model.head()

In [ ]:
# Daily return
df_gold_model["Return"] = df_gold_model["Close"].pct_change()

# Moving averages
df_gold_model["MA_7"] = df_gold_model["Close"].rolling(window=7).mean()
df_gold_model["MA_30"] = df_gold_model["Close"].rolling(window=30).mean()

# Volatility
df_gold_model["Volatility_7"] = df_gold_model["Return"].rolling(window=7).std()
df_gold_model["Volatility_30"] = df_gold_model["Return"].rolling(window=30).std()

# Momentum
df_gold_model["Momentum_7"] = df_gold_model["Close"] - df_gold_model["Close"].shift(7)
df_gold_model["Momentum_30"] = df_gold_model["Close"] - df_gold_model["Close"].shift(30)

df_gold_model.head(35)

In [ ]:
df_gold_model["Volume_MA_7"] = df_gold_model["Volume"].rolling(window=7).mean()
df_gold_model["Volume_MA_30"] = df_gold_model["Volume"].rolling(window=30).mean()

df_gold_model["Volume_Change"] = df_gold_model["Volume"].pct_change()

# Replace inf values if volume change creates division issue
df_gold_model = df_gold_model.replace([np.inf, -np.inf], np.nan)

df_gold_model.head(35)

In [ ]:
df_gold_model = df_gold_model.dropna()

print(df_gold_model.shape)
df_gold_model.head()

In [ ]:
df_gold_model.isnull().sum()

In [ ]:
forecast_horizon = 7

df_gold_model["Future_Close"] = df_gold_model["Close"].shift(-forecast_horizon)

df_gold_model["Target_Return"] = (
    df_gold_model["Future_Close"] - df_gold_model["Close"]
) / df_gold_model["Close"]

df_gold_model[["Close", "Future_Close", "Target_Return"]].head(10)

In [ ]:
df_gold_model[["Close", "Future_Close", "Target_Return"]].tail(10)

In [ ]:
df_gold_model = df_gold_model.dropna()

print(df_gold_model.shape)
df_gold_model.isnull().sum()

In [ ]:
df_gold_model["Target_Return"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_gold_model["Target_Return"], bins=50)
plt.title("Distribution of 7-Day Future Gold Returns")
plt.xlabel("7-Day Future Return")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
feature_columns_gold = df_gold_model.columns.drop(["Future_Close", "Target_Return"])

X_gold = df_gold_model[feature_columns_gold]
y_gold = df_gold_model["Target_Return"]

print("Feature columns:")
print(feature_columns_gold)

print("\nX_gold shape:", X_gold.shape)
print("y_gold shape:", y_gold.shape)

In [ ]:
train_size_gold = int(len(df_gold_model) * 0.8)

X_train_gold = X_gold.iloc[:train_size_gold]
X_test_gold = X_gold.iloc[train_size_gold:]

y_train_gold = y_gold.iloc[:train_size_gold]
y_test_gold = y_gold.iloc[train_size_gold:]

print("X_train_gold:", X_train_gold.shape)
print("X_test_gold:", X_test_gold.shape)
print("y_train_gold:", y_train_gold.shape)
print("y_test_gold:", y_test_gold.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

gold_feature_scaler = MinMaxScaler()

X_train_gold_scaled = gold_feature_scaler.fit_transform(X_train_gold)
X_test_gold_scaled = gold_feature_scaler.transform(X_test_gold)

print("X_train_gold_scaled:", X_train_gold_scaled.shape)
print("X_test_gold_scaled:", X_test_gold_scaled.shape)

print("Scaled train min:", X_train_gold_scaled.min())
print("Scaled train max:", X_train_gold_scaled.max())

In [ ]:
def create_sequences(X_data, y_data, time_steps=60):
    X_sequences = []
    y_sequences = []
    
    for i in range(time_steps, len(X_data)):
        X_sequences.append(X_data[i-time_steps:i])
        y_sequences.append(y_data[i])
    
    return np.array(X_sequences), np.array(y_sequences)

In [ ]:
time_steps = 60

X_train_gold_seq, y_train_gold_seq = create_sequences(
    X_train_gold_scaled,
    y_train_gold.values.reshape(-1, 1),
    time_steps
)

X_test_gold_seq, y_test_gold_seq = create_sequences(
    X_test_gold_scaled,
    y_test_gold.values.reshape(-1, 1),
    time_steps
)

print("X_train_gold_seq:", X_train_gold_seq.shape)
print("y_train_gold_seq:", y_train_gold_seq.shape)

print("X_test_gold_seq:", X_test_gold_seq.shape)
print("y_test_gold_seq:", y_test_gold_seq.shape)

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

In [ ]:
n_features_gold = X_train_gold_seq.shape[2]

gold_model = Sequential()

gold_model.add(LSTM(
    units=64,
    return_sequences=True,
    input_shape=(time_steps, n_features_gold)
))

gold_model.add(Dropout(0.2))

gold_model.add(LSTM(
    units=32,
    return_sequences=False
))

gold_model.add(Dropout(0.2))

gold_model.add(Dense(16, activation="relu"))

gold_model.add(Dense(1))

In [ ]:
gold_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

gold_model.summary()

In [ ]:
gold_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
gold_history = gold_model.fit(
    X_train_gold_seq,
    y_train_gold_seq,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[gold_early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gold_history.history["loss"], label="Training Loss")
plt.plot(gold_history.history["val_loss"], label="Validation Loss")
plt.title("Gold Return-Based LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("Final training loss:", gold_history.history["loss"][-1])
print("Final validation loss:", gold_history.history["val_loss"][-1])
print("Total epochs trained:", len(gold_history.history["loss"]))

In [ ]:
gold_predicted_return = gold_model.predict(X_test_gold_seq)

print("Predicted return shape:", gold_predicted_return.shape)
print("Actual return shape:", y_test_gold_seq.shape)

print("First 5 predicted returns:")
print(gold_predicted_return[:5])

print("\nFirst 5 actual returns:")
print(y_test_gold_seq[:5])

In [ ]:
current_close_gold_test = X_test_gold["Close"].iloc[time_steps:].values.reshape(-1, 1)

actual_future_price_gold = current_close_gold_test * (1 + y_test_gold_seq)
predicted_future_price_gold = current_close_gold_test * (1 + gold_predicted_return)

print("Current close shape:", current_close_gold_test.shape)
print("Actual future price shape:", actual_future_price_gold.shape)
print("Predicted future price shape:", predicted_future_price_gold.shape)

print("\nFirst 5 current close prices:")
print(current_close_gold_test[:5])

print("\nFirst 5 actual future prices:")
print(actual_future_price_gold[:5])

print("\nFirst 5 predicted future prices:")
print(predicted_future_price_gold[:5])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gold_lstm_mae = mean_absolute_error(actual_future_price_gold, predicted_future_price_gold)
gold_lstm_rmse = np.sqrt(mean_squared_error(actual_future_price_gold, predicted_future_price_gold))
gold_lstm_mape = np.mean(np.abs((actual_future_price_gold - predicted_future_price_gold) / actual_future_price_gold)) * 100
gold_lstm_r2 = r2_score(actual_future_price_gold, predicted_future_price_gold)

print("Gold Return-Based LSTM Metrics")
print("------------------------------")
print("MAE:", gold_lstm_mae)
print("RMSE:", gold_lstm_rmse)
print("MAPE:", gold_lstm_mape)
print("R² Score:", gold_lstm_r2)

In [ ]:
gold_baseline_pred = current_close_gold_test.copy()

gold_baseline_mae = mean_absolute_error(actual_future_price_gold, gold_baseline_pred)
gold_baseline_rmse = np.sqrt(mean_squared_error(actual_future_price_gold, gold_baseline_pred))
gold_baseline_mape = np.mean(np.abs((actual_future_price_gold - gold_baseline_pred) / actual_future_price_gold)) * 100
gold_baseline_r2 = r2_score(actual_future_price_gold, gold_baseline_pred)

print("Gold Naive Baseline Metrics")
print("---------------------------")
print("MAE:", gold_baseline_mae)
print("RMSE:", gold_baseline_rmse)
print("MAPE:", gold_baseline_mape)
print("R² Score:", gold_baseline_r2)

In [ ]:
gold_comparison = pd.DataFrame({
    "Model": ["Naive Baseline", "Gold Return-Based LSTM"],
    "MAE": [gold_baseline_mae, gold_lstm_mae],
    "RMSE": [gold_baseline_rmse, gold_lstm_rmse],
    "MAPE": [gold_baseline_mape, gold_lstm_mape],
    "R2": [gold_baseline_r2, gold_lstm_r2]
})

gold_comparison

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(actual_future_price_gold, label="Actual Future Gold Price")
plt.plot(predicted_future_price_gold, label="LSTM Prediction")
plt.plot(gold_baseline_pred, label="Naive Baseline", linestyle="--")

plt.title("Gold Price Forecast: Actual vs LSTM vs Naive Baseline")
plt.xlabel("Test Data Points")
plt.ylabel("Gold Price")
plt.legend()
plt.grid(True)
plt.show()

### Initial Gold LSTM Result

The return-based LSTM model was trained to predict 7-day future gold returns using 5 years of daily gold futures data.

The model outperformed the naive persistence baseline:

- Naive Baseline MAPE: 3.75%
- Gold Return-Based LSTM MAPE: 3.60%

This indicates that the LSTM was able to learn short-term patterns in gold price movement better than simply assuming the future price remains equal to the current price.

In [ ]:
external_tickers_gold = {
    "Gold": "GC=F",
    "Silver": "SI=F",
    "Crude_Oil": "CL=F",
    "Dollar_Index": "DX-Y.NYB",
    "SP500": "^GSPC",
    "NASDAQ": "^IXIC"
}

gold_market_data = {}

for name, ticker in external_tickers_gold.items():
    temp = yf.download(
        ticker,
        period="5y",
        interval="1d",
        auto_adjust=False,
        progress=False
    )
    
    if isinstance(temp.columns, pd.MultiIndex):
        temp.columns = temp.columns.get_level_values(0)
    
    gold_market_data[name] = temp

print("Downloaded data:")
for name, data in gold_market_data.items():
    print(name, data.shape)

In [ ]:
gold_ext = pd.DataFrame()

gold_ext["Gold_Open"] = gold_market_data["Gold"]["Open"]
gold_ext["Gold_High"] = gold_market_data["Gold"]["High"]
gold_ext["Gold_Low"] = gold_market_data["Gold"]["Low"]
gold_ext["Gold_Close"] = gold_market_data["Gold"]["Close"]
gold_ext["Gold_Volume"] = gold_market_data["Gold"]["Volume"]

gold_ext["Silver_Close"] = gold_market_data["Silver"]["Close"]
gold_ext["Crude_Oil_Close"] = gold_market_data["Crude_Oil"]["Close"]
gold_ext["Dollar_Index_Close"] = gold_market_data["Dollar_Index"]["Close"]
gold_ext["SP500_Close"] = gold_market_data["SP500"]["Close"]
gold_ext["NASDAQ_Close"] = gold_market_data["NASDAQ"]["Close"]

gold_ext.head()

In [ ]:
gold_ext.isnull().sum()

In [ ]:
gold_ext = gold_ext.dropna()

print(gold_ext.shape)
gold_ext.head()

In [ ]:
df_gold_ext_model = gold_ext.copy()

# Main gold return
df_gold_ext_model["Gold_Return"] = df_gold_ext_model["Gold_Close"].pct_change()

# External asset returns
df_gold_ext_model["Silver_Return"] = df_gold_ext_model["Silver_Close"].pct_change()
df_gold_ext_model["Crude_Oil_Return"] = df_gold_ext_model["Crude_Oil_Close"].pct_change()
df_gold_ext_model["Dollar_Index_Return"] = df_gold_ext_model["Dollar_Index_Close"].pct_change()
df_gold_ext_model["SP500_Return"] = df_gold_ext_model["SP500_Close"].pct_change()
df_gold_ext_model["NASDAQ_Return"] = df_gold_ext_model["NASDAQ_Close"].pct_change()

# Gold trend features
df_gold_ext_model["MA_7"] = df_gold_ext_model["Gold_Close"].rolling(7).mean()
df_gold_ext_model["MA_30"] = df_gold_ext_model["Gold_Close"].rolling(30).mean()

# Gold volatility
df_gold_ext_model["Volatility_7"] = df_gold_ext_model["Gold_Return"].rolling(7).std()
df_gold_ext_model["Volatility_30"] = df_gold_ext_model["Gold_Return"].rolling(30).std()

# Gold momentum
df_gold_ext_model["Momentum_7"] = df_gold_ext_model["Gold_Close"] - df_gold_ext_model["Gold_Close"].shift(7)
df_gold_ext_model["Momentum_30"] = df_gold_ext_model["Gold_Close"] - df_gold_ext_model["Gold_Close"].shift(30)

# Volume features
df_gold_ext_model["Volume_MA_7"] = df_gold_ext_model["Gold_Volume"].rolling(7).mean()
df_gold_ext_model["Volume_MA_30"] = df_gold_ext_model["Gold_Volume"].rolling(30).mean()
df_gold_ext_model["Volume_Change"] = df_gold_ext_model["Gold_Volume"].pct_change()

# Clean infinite values
df_gold_ext_model = df_gold_ext_model.replace([np.inf, -np.inf], np.nan)

# Drop rows created by pct_change/rolling
df_gold_ext_model = df_gold_ext_model.dropna()

print(df_gold_ext_model.shape)
df_gold_ext_model.head()

In [ ]:
forecast_horizon = 7

df_gold_ext_model["Future_Gold_Close"] = df_gold_ext_model["Gold_Close"].shift(-forecast_horizon)

df_gold_ext_model["Target_Return"] = (
    df_gold_ext_model["Future_Gold_Close"] - df_gold_ext_model["Gold_Close"]
) / df_gold_ext_model["Gold_Close"]

df_gold_ext_model = df_gold_ext_model.dropna()

print(df_gold_ext_model.shape)
df_gold_ext_model[["Gold_Close", "Future_Gold_Close", "Target_Return"]].head()

In [ ]:
df_gold_ext_model["Target_Return"].describe()

In [ ]:
gold_ext_feature_columns = df_gold_ext_model.columns.drop(
    ["Future_Gold_Close", "Target_Return"]
)

X_gold_ext = df_gold_ext_model[gold_ext_feature_columns]
y_gold_ext = df_gold_ext_model["Target_Return"]

print("Number of features:", len(gold_ext_feature_columns))
print("X_gold_ext shape:", X_gold_ext.shape)
print("y_gold_ext shape:", y_gold_ext.shape)

print(gold_ext_feature_columns)

In [ ]:
train_size_gold_ext = int(len(df_gold_ext_model) * 0.8)

X_train_gold_ext = X_gold_ext.iloc[:train_size_gold_ext]
X_test_gold_ext = X_gold_ext.iloc[train_size_gold_ext:]

y_train_gold_ext = y_gold_ext.iloc[:train_size_gold_ext]
y_test_gold_ext = y_gold_ext.iloc[train_size_gold_ext:]

print("X_train_gold_ext:", X_train_gold_ext.shape)
print("X_test_gold_ext:", X_test_gold_ext.shape)
print("y_train_gold_ext:", y_train_gold_ext.shape)
print("y_test_gold_ext:", y_test_gold_ext.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

gold_ext_feature_scaler = MinMaxScaler()

X_train_gold_ext_scaled = gold_ext_feature_scaler.fit_transform(X_train_gold_ext)
X_test_gold_ext_scaled = gold_ext_feature_scaler.transform(X_test_gold_ext)

print("X_train_gold_ext_scaled:", X_train_gold_ext_scaled.shape)
print("X_test_gold_ext_scaled:", X_test_gold_ext_scaled.shape)

print("Scaled train min:", X_train_gold_ext_scaled.min())
print("Scaled train max:", X_train_gold_ext_scaled.max())

In [ ]:
time_steps = 60

X_train_gold_ext_seq, y_train_gold_ext_seq = create_sequences(
    X_train_gold_ext_scaled,
    y_train_gold_ext.values.reshape(-1, 1),
    time_steps
)

X_test_gold_ext_seq, y_test_gold_ext_seq = create_sequences(
    X_test_gold_ext_scaled,
    y_test_gold_ext.values.reshape(-1, 1),
    time_steps
)

print("X_train_gold_ext_seq:", X_train_gold_ext_seq.shape)
print("y_train_gold_ext_seq:", y_train_gold_ext_seq.shape)

print("X_test_gold_ext_seq:", X_test_gold_ext_seq.shape)
print("y_test_gold_ext_seq:", y_test_gold_ext_seq.shape)

In [ ]:
n_features_gold_ext = X_train_gold_ext_seq.shape[2]

gold_ext_model = Sequential()

gold_ext_model.add(LSTM(
    units=64,
    return_sequences=True,
    input_shape=(time_steps, n_features_gold_ext)
))

gold_ext_model.add(Dropout(0.2))

gold_ext_model.add(LSTM(
    units=32,
    return_sequences=False
))

gold_ext_model.add(Dropout(0.2))

gold_ext_model.add(Dense(16, activation="relu"))

gold_ext_model.add(Dense(1))

In [ ]:
gold_ext_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

gold_ext_model.summary()

In [ ]:
gold_ext_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [ ]:
gold_ext_history = gold_ext_model.fit(
    X_train_gold_ext_seq,
    y_train_gold_ext_seq,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[gold_ext_early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gold_ext_history.history["loss"], label="Training Loss")
plt.plot(gold_ext_history.history["val_loss"], label="Validation Loss")
plt.title("Gold External-Features LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("Final training loss:", gold_ext_history.history["loss"][-1])
print("Final validation loss:", gold_ext_history.history["val_loss"][-1])
print("Total epochs trained:", len(gold_ext_history.history["loss"]))

In [ ]:
gold_ext_predicted_return = gold_ext_model.predict(X_test_gold_ext_seq)

print("Predicted return shape:", gold_ext_predicted_return.shape)
print("Actual return shape:", y_test_gold_ext_seq.shape)

print("First 5 predicted returns:")
print(gold_ext_predicted_return[:5])

print("\nFirst 5 actual returns:")
print(y_test_gold_ext_seq[:5])

In [ ]:
current_close_gold_ext_test = X_test_gold_ext["Gold_Close"].iloc[time_steps:].values.reshape(-1, 1)

actual_future_price_gold_ext = current_close_gold_ext_test * (1 + y_test_gold_ext_seq)
predicted_future_price_gold_ext = current_close_gold_ext_test * (1 + gold_ext_predicted_return)

print("Current close shape:", current_close_gold_ext_test.shape)
print("Actual future price shape:", actual_future_price_gold_ext.shape)
print("Predicted future price shape:", predicted_future_price_gold_ext.shape)

In [ ]:
gold_ext_lstm_mae = mean_absolute_error(
    actual_future_price_gold_ext,
    predicted_future_price_gold_ext
)

gold_ext_lstm_rmse = np.sqrt(mean_squared_error(
    actual_future_price_gold_ext,
    predicted_future_price_gold_ext
))

gold_ext_lstm_mape = np.mean(
    np.abs((actual_future_price_gold_ext - predicted_future_price_gold_ext) / actual_future_price_gold_ext)
) * 100

gold_ext_lstm_r2 = r2_score(
    actual_future_price_gold_ext,
    predicted_future_price_gold_ext
)

print("Gold External-Features LSTM Metrics")
print("-----------------------------------")
print("MAE:", gold_ext_lstm_mae)
print("RMSE:", gold_ext_lstm_rmse)
print("MAPE:", gold_ext_lstm_mape)
print("R² Score:", gold_ext_lstm_r2)

In [ ]:
gold_ext_baseline_pred = current_close_gold_ext_test.copy()

gold_ext_baseline_mae = mean_absolute_error(
    actual_future_price_gold_ext,
    gold_ext_baseline_pred
)

gold_ext_baseline_rmse = np.sqrt(mean_squared_error(
    actual_future_price_gold_ext,
    gold_ext_baseline_pred
))

gold_ext_baseline_mape = np.mean(
    np.abs((actual_future_price_gold_ext - gold_ext_baseline_pred) / actual_future_price_gold_ext)
) * 100

gold_ext_baseline_r2 = r2_score(
    actual_future_price_gold_ext,
    gold_ext_baseline_pred
)

print("Gold External Naive Baseline Metrics")
print("------------------------------------")
print("MAE:", gold_ext_baseline_mae)
print("RMSE:", gold_ext_baseline_rmse)
print("MAPE:", gold_ext_baseline_mape)
print("R² Score:", gold_ext_baseline_r2)

In [ ]:
gold_final_comparison = pd.DataFrame({
    "Model": [
        "Gold Naive Baseline",
        "Gold Market-Only LSTM",
        "Gold External-Features Naive Baseline",
        "Gold External-Features LSTM"
    ],
    "MAE": [
        gold_baseline_mae,
        gold_lstm_mae,
        gold_ext_baseline_mae,
        gold_ext_lstm_mae
    ],
    "RMSE": [
        gold_baseline_rmse,
        gold_lstm_rmse,
        gold_ext_baseline_rmse,
        gold_ext_lstm_rmse
    ],
    "MAPE": [
        gold_baseline_mape,
        gold_lstm_mape,
        gold_ext_baseline_mape,
        gold_ext_lstm_mape
    ],
    "R2": [
        gold_baseline_r2,
        gold_lstm_r2,
        gold_ext_baseline_r2,
        gold_ext_lstm_r2
    ]
})

gold_final_comparison

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(actual_future_price_gold_ext, label="Actual Future Gold Price")
plt.plot(predicted_future_price_gold_ext, label="External-Features LSTM")
plt.plot(gold_ext_baseline_pred, label="Naive Baseline", linestyle="--")

plt.title("Gold Forecast: Actual vs External-Features LSTM vs Naive Baseline")
plt.xlabel("Test Data Points")
plt.ylabel("Gold Price")
plt.legend()
plt.grid(True)
plt.show()

### Model Selection Conclusion

The Gold Market-Only Return-Based LSTM performed best among the tested models. Although external market indicators such as silver, crude oil, dollar index, S&P 500, and NASDAQ were added, the external-feature model showed signs of overfitting and underperformed the simpler market-only LSTM.

Final selected model:

Gold Market-Only Return-Based LSTM

Reason:

- It achieved the lowest MAPE: approximately 3.60%
- It outperformed the naive persistence baseline
- It avoided the overfitting seen in the external-feature model

In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

gold_model.save("models/gold_market_only_lstm.keras")

joblib.dump(gold_feature_scaler, "models/gold_feature_scaler.pkl")
joblib.dump(list(feature_columns_gold), "models/gold_feature_columns.pkl")
joblib.dump(time_steps, "models/gold_time_steps.pkl")

gold_comparison.to_csv("outputs/gold_model_comparison.csv", index=False)

print("Gold market-only model and files saved successfully.")
print("Models folder:", os.listdir("models"))
print("Outputs folder:", os.listdir("outputs"))

In [ ]:
latest_gold_data = X_gold[feature_columns_gold].tail(time_steps)

print("Latest gold data shape:", latest_gold_data.shape)
latest_gold_data.tail()

In [ ]:
latest_gold_scaled = gold_feature_scaler.transform(latest_gold_data)

latest_gold_seq = latest_gold_scaled.reshape(
    1,
    time_steps,
    len(feature_columns_gold)
)

print("Latest gold sequence shape:", latest_gold_seq.shape)

In [ ]:
latest_gold_predicted_return = gold_model.predict(latest_gold_seq, verbose=0)[0][0]

print("Predicted 7-day gold return:", latest_gold_predicted_return)
print("Predicted 7-day gold return (%):", latest_gold_predicted_return * 100)

In [ ]:
current_gold_price = df_gold_model["Close"].iloc[-1]
predicted_gold_future_price = current_gold_price * (1 + latest_gold_predicted_return)

print("Current Gold Price:", current_gold_price)
print("Predicted 7-Day Future Gold Price:", predicted_gold_future_price)
print("Expected Change:", predicted_gold_future_price - current_gold_price)
print("Expected Change (%):", latest_gold_predicted_return * 100)

In [ ]:
def classify_gold_risk(predicted_return):
    if predicted_return >= 0.02:
        return "Bullish / High Upside"
    elif predicted_return >= 0.005:
        return "Moderately Bullish"
    elif predicted_return <= -0.02:
        return "Bearish / Downside Risk"
    elif predicted_return <= -0.005:
        return "Moderately Bearish"
    else:
        return "Neutral"

In [ ]:
latest_gold_risk = classify_gold_risk(latest_gold_predicted_return)

print("Gold Risk Signal:", latest_gold_risk)

In [ ]:
def get_gold_recommendation(risk_signal):
    if risk_signal == "Bullish / High Upside":
        return "Model indicates strong short-term upside; monitor macro news and avoid aggressive short positions."
    elif risk_signal == "Moderately Bullish":
        return "Model indicates mild upside; continue monitoring dollar index, inflation, and rate-related news."
    elif risk_signal == "Bearish / Downside Risk":
        return "Model indicates downside risk; monitor dollar strength, rate expectations, and profit-booking pressure."
    elif risk_signal == "Moderately Bearish":
        return "Model indicates mild downside; avoid overexposure and wait for confirmation."
    else:
        return "Model indicates limited movement; continue normal monitoring."

In [ ]:
latest_gold_recommendation = get_gold_recommendation(latest_gold_risk)

print("Recommendation:", latest_gold_recommendation)

In [ ]:
latest_gold_prediction_summary = {
    "Current Gold Price": round(float(current_gold_price), 2),
    "Predicted 7-Day Future Gold Price": round(float(predicted_gold_future_price), 2),
    "Expected Change": round(float(predicted_gold_future_price - current_gold_price), 2),
    "Expected Change (%)": round(float(latest_gold_predicted_return * 100), 4),
    "Risk Signal": latest_gold_risk,
    "Recommendation": latest_gold_recommendation
}

latest_gold_prediction_summary

In [ ]:
def predict_latest_gold_price(
    model,
    scaler,
    feature_data,
    feature_columns,
    time_steps=60
):
    latest_data = feature_data[feature_columns].tail(time_steps)

    latest_scaled = scaler.transform(latest_data)

    latest_seq = latest_scaled.reshape(
        1,
        time_steps,
        len(feature_columns)
    )

    predicted_return = model.predict(latest_seq, verbose=0)[0][0]

    current_price = feature_data["Close"].iloc[-1]
    predicted_future_price = current_price * (1 + predicted_return)

    expected_change = predicted_future_price - current_price
    expected_change_percent = predicted_return * 100

    risk_signal = classify_gold_risk(predicted_return)
    recommendation = get_gold_recommendation(risk_signal)

    prediction_summary = {
        "Current Gold Price": round(float(current_price), 2),
        "Predicted 7-Day Future Gold Price": round(float(predicted_future_price), 2),
        "Expected Change": round(float(expected_change), 2),
        "Expected Change (%)": round(float(expected_change_percent), 4),
        "Risk Signal": risk_signal,
        "Recommendation": recommendation
    }

    return prediction_summary

In [ ]:
latest_gold_prediction_summary = predict_latest_gold_price(
    model=gold_model,
    scaler=gold_feature_scaler,
    feature_data=df_gold_model,
    feature_columns=feature_columns_gold,
    time_steps=time_steps
)

latest_gold_prediction_summary

In [ ]:
import json

with open("outputs/latest_gold_prediction_summary.json", "w") as f:
    json.dump(latest_gold_prediction_summary, f, indent=4)

print("Latest gold prediction summary saved successfully.")
print("Outputs folder:", os.listdir("outputs"))